In [1]:
from s2_simu_mcmc_sur import *

In [2]:
# 1 #####################################################################################################
# Set-up
target = 'm_sur'

# initializing models
IPM_true = GP_IPM(popu_data=popu_dataset, GPmodel_true=models_true)
IPM_pret_sur = Perted_IPM(popu_data=popu_dataset, GPmodel_true=models_true, target=target, NB=False)

# we build the GPR model.
l = gpflow.likelihoods.Bernoulli()
kernel = gpflow.kernels.RBF(lengthscales=np.array([10,10])) 
m_sur_new = gpflow.models.GPMC(data=(IPM_pret_sur.X_sur, IPM_pret_sur.Y_sur), kernel=kernel, likelihood=l)

m_sur_new.kernel.lengthscales.prior = tfd.Normal(loc=f64(0.), scale=f64(100.))
m_sur_new.kernel.variance.prior = tfd.InverseGamma(f64(0.001),f64(0.001))
#########################################################################################################

In [ ]:
# 2 #####################################################################################################
# We now read samples
hmc_helper = gpflow.optimizers.SamplingHelper(
    m_sur_new.log_posterior_density, m_sur_new.trainable_parameters
)
num_burnin_steps = 30000
num_samples = 20000

# samples = pickle.load(open(file = os.getcwd()+"/mcmc_samples/m_sur" +"/samples.pkl", mode="rb"))
parameter_samples = pickle.load(open(file = os.getcwd()+"/mcmc_samples/m_sur" +"/parameter_samples.pkl", mode="rb"))

#  assign samples to our models
IPM_pret_sur.mcmc_para_sample = parameter_samples

#########################################################################################################

In [4]:
# 3 #####################################################################################################
# loading populations which were generated by MCMC samples
print('Loading summary_data for all the MCMC samples')
summary_data = pickle.load(open(file = os.getcwd() + "/mcmc_samples/m_sur" + "/summary_data.pkl", mode="rb"))
IPM_pret_sur.nlog_post = np.array(summary_data['nlpo'])
IPM_pret_sur.nlog_likeli = np.array(summary_data['nll'])

#########################################################################################################

# 4 #####################################################################################################
# Now, if we consider the estimates with the likelihood scores around the optimum.
#      Calculating the summary tests around the optimum for full data.
rep = 1000
opt_percentage = 2
print('\nLoading summary_data for all the MCMC samples around the optimum')
summary_opt = pickle.load(open(file = os.getcwd() + "/mcmc_samples/m_sur" + "/summary_opt.pkl", mode="rb"))
summary_around_opt = pickle.load(open(file = os.getcwd() + "/mcmc_samples/m_sur" + "/summary_around_opt.pkl", mode="rb"))


Loading summary_data for all the MCMC samples

Loading summary_data for all the MCMC samples around the optimum


In [7]:
# 5 #####################################################################################################
# find the top ten stats which are most sens for this kind of perturbation
IPM_pret_sur.opt_percentage=opt_percentage
num_mcmc = IPM_pret_sur.mcmc_para_sample[0].shape[0]

most_freq_summary_stats = pd.DataFrame(data=0.0, index=range(1), columns=IPM_pret_sur.col_names)

for j in range(int(IPM_pret_sur.mcmc_para_sample[0].shape[0]*opt_percentage/100)):
    d = summary_around_opt.loc[(0+j*rep):(rep-1+j*rep)].reset_index(drop=True).copy()
    auc0 = np.zeros(76)
    auc1 = np.zeros(76)
    for i in range(76):        
        fpr0, tpr0, _ = roc_curve(y_true=np.append(np.repeat(1, rep), np.repeat(0, rep)), 
                                    y_score=np.append(summary_opt.iloc[:, i], d.iloc[:, i]), pos_label=0)
        auc0[i] = auc(fpr0, tpr0)
    
    most_freq_summary_stats.iloc[:, np.argsort(auc0)[np.sort(auc0) > 0.75][-10:]] += 1

In [8]:
print(np.sort(most_freq_summary_stats.loc[0])[-10:])
print(most_freq_summary_stats.columns[np.argsort(most_freq_summary_stats.loc[0])[-10:]]) 

[257. 317. 338. 340. 385. 389. 391. 394. 394. 399.]
Index(['23b', '5a', '6a', '14b', '14a', '13a', '24b', '15b', '13b', '15a'], dtype='object')
